## 3. Custom Feature Engineering & Preprocessing Pipeline

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class HouseFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        # Combined Square Footage
        X_copy['TotalSF'] = X_copy['TotalBsmtSF'].fillna(0) + X_copy['1stFlrSF'].fillna(0) + X_copy['2ndFlrSF'].fillna(0)
        # Mathematically Corrected Bathrooms calculation
        X_copy['TotalBath'] = (
            X_copy['FullBath'].fillna(0) +
            (0.5 * X_copy['HalfBath'].fillna(0)) +
            X_copy['BsmtFullBath'].fillna(0) +
            (0.5 * X_copy['BsmtHalfBath'].fillna(0))
        )
        # Property age at sale time
        X_copy['HouseAge'] = X_copy['YrSold'] - X_copy['YearBuilt']
        return X_copy

print("HouseFeatureEngineer class successfully declared.")

## 4. Exploratory Data Diagnostics (Mutual Information on Development Partition)

In [ ]:
X_mi = X_dev.copy()
discrete_mask = []
for colname in X_mi.columns:
    if colname == 'MSSubClass' or X_mi[colname].dtype == 'object':
        X_mi[colname], _ = X_mi[colname].factorize()
        discrete_mask.append(True)
    else:
        discrete_mask.append(False)

X_mi = X_mi.fillna(0)
mi_scores = mutual_info_regression(X_mi, y_dev, discrete_features=discrete_mask, random_state=42)
mi_series = pd.Series(mi_scores, index=X_mi.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=mi_series.values[:20], y=mi_series.index[:20], hue=mi_series.index[:20], palette='viridis', legend=False)
plt.title("Top 20 Features by Mutual Information (Development Split Only)")
plt.xlabel("Mutual Information Score")
plt.show()

## 6. Fair Feature-Selection Comparison

In [ ]:
def custom_mi_scorer(X, y):
    n_num = len(numerical_cols)
    discrete_features = [False] * n_num + [True] * (X.shape[1] - n_num)
    return mutual_info_regression(X, y, discrete_features=discrete_features, random_state=42)

selection_mi = SelectPercentile(score_func=custom_mi_scorer, percentile=80)

base_model = models[screening_winner_name]

pipe_passthrough = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor)),
    ('model', clone(base_model))
])

pipe_mi = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor)),
    ('selection', selection_mi),
    ('model', clone(base_model))
])

pipe_lasso = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor)),
    ('selection', clone(selection_step)),
    ('model', clone(base_model))
])

# Dedicated Feature Selection Evaluation (Evaluated into its own dictionary so as not to clutter the model tournament table)
feature_selection_results = {}

def evaluate_feature_selector(name, estimator):
    scores = cross_validate(clone(estimator), X_dev, y_dev, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
    mae_mean = -scores['test_MAE'].mean()
    mae_std = scores['test_MAE'].std()
    mse_mean = -scores['test_MSE'].mean()
    rmse_mean = -scores['test_RMSE'].mean()
    r2_mean = scores['test_R2'].mean()
    feature_selection_results[name] = {
        'Feature Selection Scheme': name,
        'CV MAE Mean': mae_mean,
        'CV MAE Std': mae_std,
        'CV MSE Mean': mse_mean,
        'CV RMSE Mean': rmse_mean,
        'CV R2 Mean': r2_mean
    }
    print(f"Evaluated: {name} | CV MAE: ${mae_mean:,.2f} (Std: ${mae_std:,.2f})")

print("=== COMPARING SELECTION SCHEMES ===")
evaluate_feature_selector("No Selection (Passthrough)", pipe_passthrough)
evaluate_feature_selector("MI Selection (Percentile)", pipe_mi)
evaluate_feature_selector("Lasso Selection (Model-Based)", pipe_lasso)

selector_choices = {
    "No Selection (Passthrough)": 'passthrough',
    "MI Selection (Percentile)": selection_mi,
    "Lasso Selection (Model-Based)": selection_step
}

chosen_selector_name = min(
    ["No Selection (Passthrough)", "MI Selection (Percentile)", "Lasso Selection (Model-Based)"],
    key=lambda x: feature_selection_results[x]['CV MAE Mean']
)
chosen_selector = selector_choices[chosen_selector_name]
print(f"Selected Feature Selector: {chosen_selector_name}")

pip_fit = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', clone(preprocessor))
])
X_preprocessed = pip_fit.fit_transform(X_dev, y_dev)
all_features = pip_fit.named_steps['preprocessor'].get_feature_names_out()

lasso_fit = Lasso(alpha=100, max_iter=10000, tol=0.01, random_state=42)
lasso_fit.fit(X_preprocessed, y_dev)

coef_df = pd.DataFrame({
    'Feature': [feat.split('__')[-1] for feat in all_features],
    'Coefficient': lasso_fit.coef_,
    'AbsCoef': np.abs(lasso_fit.coef_)
}).sort_values(by='AbsCoef', ascending=False).head(30)
coef_df = coef_df[coef_df['AbsCoef'] > 0]

plt.figure(figsize=(10, 8))
sns.barplot(x='AbsCoef', y='Feature', data=coef_df, palette='magma', hue='Feature', legend=False)
plt.title("Top Predictive Features Selected by Lasso (Absolute Coefficients)")
plt.xlabel("Lasso Coefficient Weight Magnitude")
plt.ylabel("Feature Name")
plt.show()